In [ ]:
import SimpleITK as sitk
import numpy as np
import os
import matplotlib.pyplot as plt
from ipywidgets import interact, fixed
from IPython.display import clear_output
import math

In [ ]:
# Callback invoked by the IPython interact method for scrolling through image stacks of
# the two images being registered.
def display_images(fixed_image, moving_image):
    # Create a figure with two subplots and the specified size.
    fig, ax = plt.subplots()
    
    # 1. Plot the bottom image
    ax.imshow(fixed_image, cmap='gray')
    # 2. Plot the top image with transparency (alpha between 0 and 1)
    ax.imshow(moving_image, cmap='jet', alpha=0.5)
    
    plt.axis('on') # Optional: Hide the axis lines and labels
    plt.show()

# Callback invoked when the StartEvent happens, sets up our new data.
def start_plot():
    global metric_values, multires_iterations

    metric_values = []
    multires_iterations = []


# Callback invoked when the EndEvent happens, do cleanup of data and figure.
def end_plot():
    global metric_values, multires_iterations

    del metric_values
    del multires_iterations
    # Close figure, we don't want to get a duplicate of the plot latter on.
    plt.close()


# Callback invoked when the IterationEvent happens, update our data and display new figure.
def plot_values(registration):
    global metric_values, multires_iterations

    metric_values.append(registration.GetMetricValue())
    # Clear the output area (wait=True, to reduce flickering), and plot current data
    clear_output(wait=True)
    # Plot the similarity metric values
    plt.plot(metric_values, "r")
    plt.plot(
        multires_iterations,
        [metric_values[index] for index in multires_iterations],
        "b*",
    )
    plt.xlabel("Iteration Number", fontsize=12)
    plt.ylabel("Metric Value", fontsize=12)
    plt.show()


# Callback invoked when the sitkMultiResolutionIterationEvent happens, update the index into the
# metric_values list.
def update_multires_iterations():
    global metric_values, multires_iterations
    multires_iterations.append(len(metric_values))

In [ ]:
animal = 'DK37'
moving_index = '472'
fixed_index = '473'
tif_path = f'/net/birdstore/Active_Atlas_Data/data_root/pipeline_data/{animal}/preps/C1/thumbnail_cleaned'
fixed_image_path = os.path.join(tif_path, f'{fixed_index}.tif')
moving_image_path = os.path.join(tif_path, f'{moving_index}.tif')
fixed_image = sitk.ReadImage(fixed_image_path, sitk.sitkFloat32)
moving_image = sitk.ReadImage(moving_image_path, sitk.sitkFloat32)

In [ ]:
print('fixed spacing', fixed_image.GetSpacing())
print('moving spacing', moving_image.GetSpacing())

In [ ]:
transform_method = sitk.AffineTransform(2)
initial_transform = sitk.CenteredTransformInitializer(
    fixed_image,
    moving_image,
    transform_method,
    sitk.CenteredTransformInitializerFilter.GEOMETRY,
)

In [ ]:
%%time

# Compute landmark transform

registration = sitk.ImageRegistrationMethod()
# Similarity metric settings.
#registration.SetMetricAsCorrelation()
#registration.SetMetricAsJointHistogramMutualInformation()
#registration.SetMetricAsMeanSquares()
registration.SetMetricAsMattesMutualInformation()
registration.SetMetricSamplingStrategy(registration.RANDOM)
registration.SetMetricSamplingPercentage(0.1)
registration.SetInterpolator(sitk.sitkLinear)
# Optimizer settings.
registration.SetOptimizerAsGradientDescent(
    learningRate=1,
    numberOfIterations=150,
    convergenceMinimumValue=1e-6,
    convergenceWindowSize=10)
registration.SetOptimizerScalesFromPhysicalShift()
registration.SetShrinkFactorsPerLevel(shrinkFactors=[4, 2, 1])
registration.SetSmoothingSigmasPerLevel(smoothingSigmas=[2, 1, 0])
registration.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()
registration.SetInitialTransform(initial_transform, inPlace=False)

# Connect all of the observers so that we can perform plotting during registration.
registration.AddCommand(sitk.sitkStartEvent, start_plot)
registration.AddCommand(sitk.sitkEndEvent, end_plot)
registration.AddCommand(sitk.sitkMultiResolutionIterationEvent, update_multires_iterations)
registration.AddCommand(sitk.sitkIterationEvent, lambda: plot_values(registration))

final_transform = registration.Execute(fixed_image, moving_image)

In [ ]:
print(f"Final metric value: {registration.GetMetricValue()}")
print(f"Optimizer's stopping condition, {registration.GetOptimizerStopConditionDescription()}")
print(f'final params for moving index {moving_index} {final_transform.GetParameters()}')

In [ ]:
moving_resampled = sitk.Resample(
    moving_image,
    fixed_image,
    final_transform,
    sitk.sitkLinear,
    0.0,
    moving_image.GetPixelID(),
)

In [ ]:
display_images(fixed_image, moving_resampled)